# Identifying Causality Without Randomization: Regression Discontinuity and Synthetic Control

Causal inference ideally relies on randomized experiments, but the reality of social science is that treatment is often not randomly assigned: subsidies are distributed based on a score cutoff, a city unilaterally raises taxes. In such cases we settle for the next best thing, looking for "near-random" gaps to squeeze out causal effects from observational data using quasi-experimental designs. This tutorial covers two of the most widely used approaches, each exploiting a different type of such gap.

The first is **Regression Discontinuity (RDD)**. When treatment is triggered by a sharp rule—a score ≥ 60 triggers admission, income ≤ threshold qualifies for subsidy—individuals just on either side of the threshold are effectively "randomly" assigned to treatment and control: someone scoring 59.9 versus 60.1 differs only in their assignment to one side of the cutoff, not in underlying ability. Thus the **jump** in the outcome at the discontinuity is the local causal effect (LATE) caused by that rule. The key difficulty is this: the outcome curve outside the threshold neighborhood may be curved or sloped, and a crude global line will mistake that curvature for a jump. The correct approach is to fit only **locally** near the discontinuity using local linear regression.

The second is **Synthetic Control (SCM)**. Some treatments naturally affect only **one** unit: a country changes its law, a city hosts an expo. There is no second identical country to serve as a control. Synthetic Control's solution is to construct a "counterfactual, had this unit not been treated, what would have happened" using a **weighted combination** of untreated "donor" units; after treatment, the gap between the true trajectory and the synthetic trajectory is the effect. Its challenge lies in the credibility of the counterfactual—only when the synthetic control closely tracked the treated unit **before** treatment can the gap **after** treatment be called an effect.

We demonstrate on two built-in toy datasets. Both are generated from real data-generating processes with **known parameters** (RDD's true jump τ = 3.0, the panel's true treatment effect att = −0.8), so by the end of the tutorial you can see with your own eyes whether the estimators **recover the known true values**. The entire workflow uses `socialverse`, a library for social science analysis modeled after R's `rdrobust`, `Synth` / `gsynth` and other specialized packages; one additional thing it does, we save for the final section.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件

import os
import sys

# 让 notebook 无论从哪启动都加载 worktree 里的 socialverse,并把图存到本目录。
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:  # 交互式内核里没有 __file__
    _here = os.getcwd()
_root = os.path.dirname(_here)
if os.path.isdir(os.path.join(_root, "socialverse")) and _root not in sys.path:
    sys.path.insert(0, _root)
if os.path.isdir(_here):
    os.chdir(_here)

import numpy as np
import pandas as pd

import socialverse as sv
from socialverse import datasets as ds

---
# Part I: Regression Discontinuity

## Loading RDD Data

`ds.load_rdd(tau=3.0)` generates a sharp regression discontinuity dataset: the running variable triggers treatment at `cutoff=0`, with true jump `τ = 3.0`. The four columns are `running` (the running variable determining treatment), `treat` (0/1 treatment indicator), `y` (outcome), and `x` (a covariate). We intentionally embedded slope and curvature in the outcome (`1.5·running − 0.8·running²`) precisely to force local fitting near the discontinuity—otherwise a crude global line would be biased by this curvature.

In [2]:
rdd_df = ds.load_rdd(tau=3.0)
rdd_df.head()

,running,treat,y,x
0,0.2739,1,5.9588,1.3554
1,-0.4604,0,0.9777,0.0022
2,-0.9181,0,-0.0898,-0.7905
3,-0.9669,0,-1.0274,0.1419
4,0.6265,1,5.4718,0.2176


Check the sample sizes on either side of the cutoff: there are several hundred observations to the left and right of cutoff=0, sufficient for local fitting in the discontinuity neighborhood.

In [3]:
n_left = int((rdd_df["running"] < 0).sum())
n_right = int((rdd_df["running"] >= 0).sum())
print(f"样本量 = {len(rdd_df)}  |  cutoff=0 左侧 {n_left}  右侧 {n_right}")

样本量 = 500  |  cutoff=0 左侧 223  右侧 277


## Declaring Outcome and Estimand

Before estimation, record which column is the outcome and what we are estimating in the research state. Regression discontinuity identifies the **local** average treatment effect (LATE) at the cutoff, so set `estimand.target` to `LATE`; set the outcome variable to `y`; and add the data itself to the state. Subsequent estimation functions will read their settings from here, requiring no repeated parameter passing.

In [4]:
st = sv.StudyState()
st.write("estimand", "target", "LATE")   # 断点回归识别的是局部平均处理效应
st.write("variables", "outcome", "y")    # 结果变量
st.write("sources", "datasets", rdd_df)  # 数据本身
st

StudyState(sources[1], variables[1], estimand[1]; 0 steps)

## Estimating the Discontinuity Jump

`sv.tl.rdd` does the following concretely: first, use a data-driven heuristic (IK-lite) to choose a bandwidth `h`, keeping only observations within `±h` of the cutoff; then run separate **triangular kernel-weighted** local linear regressions on either side of the cutoff, with weights smoothly declining from 1 at the cutoff to 0 at the bandwidth edge, focusing estimation in the discontinuity neighborhood; read off the fitted line intercepts on each side at the cutoff, and **right intercept − left intercept** is the jump. It updates `st` in place, writing the jump, standard error, and intercepts on both sides to `models.rdd`, and the bandwidth and effective sample sizes to `diagnostics.bandwidth`.

In [5]:
sv.tl.rdd(st, running="running", cutoff=0.0)  # running 变量列名 + 断点位置

model = st.models["rdd"]
bw = st.diagnostics["bandwidth"]
print(f"真实跳跃 τ  = 3.0   (数据生成过程已知)")
print(f"估计跳跃 τ̂  = {model['jump']:.4f}   SE={model['se']:.4f}  t={model['t']:.2f}")
print(f"  左侧断点截距 = {model['left_intercept']:.4f}")
print(f"  右侧断点截距 = {model['right_intercept']:.4f}")
print(f"带宽 h = {bw['h']:.4f} ({bw['selector']})  |  左侧 n={bw['n_left']}  右侧 n={bw['n_right']}")

真实跳跃 τ  = 3.0   (数据生成过程已知)
估计跳跃 τ̂  = 2.9926   SE=0.1015  t=29.47
  左侧断点截距 = 1.9867
  右侧断点截距 = 4.9793
带宽 h = 0.7650 (IK-lite rule-of-thumb)  |  左侧 n=171  右侧 n=210


The estimated jump is `τ̂ ≈ 2.99`, nearly identical to the true `3.0`, with standard error around 0.10 and t-statistic as high as 29—the outcome clearly has a significant and correctly-sized jump at the cutoff. The difference between the right and left discontinuity intercepts (approximately 4.98 and 1.99) is exactly this jump. Local linear fitting with triangular kernel keeps focus on the discontinuity neighborhood, avoiding the confounding effects of distant curvature, which is precisely the core insight of `rdrobust`.

## Discontinuity Plot

Beyond numbers, a discontinuity plot reveals at a glance whether a jump truly exists. `sv.pl.rdd_plot` reads `models.rdd`, bins the running variable into equal-count bins on each side, plots the mean outcome for each bin as a scatter point, and overlays the local linear fit lines on both sides; the vertical gap between these two lines at the cutoff is the estimated jump. `out=` saves the plot directly as a PNG in the same directory.

In [6]:
sv.pl.rdd_plot(st, out="fig_rdd.png")
print("已存:", st.artifacts["figures"]["rdd"]["path"])

已存: fig_rdd.png


![RDD discontinuity plot](fig_rdd.png)

The binned means on the left (blue) and right (green) each fit a local linear line; their vertical gap at `running=0` (the red dashed line) is approximately 3, exactly our estimated causal jump. The curves on either side extend smoothly and break only at the cutoff, indicating this gap is not a curvature artifact.

---
# Part II: Synthetic Control

## Loading Panel Data

`ds.load_did_panel(att=-0.8)` is a panel with **parallel pre-treatment trends**: 40 firms × 8 years (2010–2017), with the first half treated in 2015, and true treatment effect `att = −0.8`. Columns include unit `firm_id`, time `year`, treatment group indicator `treat`, outcome `y`, and more. Synthetic Control is designed for the "single treated unit" scenario, so we isolate just **one** treated unit (`firm_id=0`) and construct a counterfactual for it using a weighted combination of the remaining donor units.

In [7]:
panel = ds.load_did_panel(att=-0.8)
panel.head()

,firm_id,year,treat,post,treat_post,first_treated,y,x1
0,0,2010,1,0,0,2015,1.6221,1.5139
1,0,2011,1,0,0,2015,2.8355,0.7813
2,0,2012,1,0,0,2015,2.5744,-0.3139
3,0,2013,1,0,0,2015,3.2232,1.9603
4,0,2014,1,0,0,2015,3.5321,1.3151


Confirm the panel dimensions and treatment timing: 40 units, 8 years, treatment begins in 2015.

In [8]:
print("单位数:", panel["firm_id"].nunique(),
      "|  年份:", f"{panel['year'].min()}–{panel['year'].max()}",
      "|  处理起始年: 2015")

单位数: 40 |  年份: 2010–2017 |  处理起始年: 2015


## Declaring Design and Estimand

Synthetic Control requires two additional declarations: which column marks units that were ever treated (so we can exclude treated units from the donor pool, avoiding counterfactual contamination), and which column is the time index. So beyond outcome `y` and data, add `design.treatment` and `design.time` to the state. Note the estimand target here is **ATT** (average treatment effect on the treated), not the LATE of regression discontinuity.

In [9]:
st2 = sv.StudyState()
st2.write("design", "treatment", "treat")   # 用来把「曾被处理」的单位踢出捐赠池
st2.write("design", "time", "year")         # 时间轴
st2.write("variables", "outcome", "y")      # 结果变量
st2.write("estimand", "target", "ATT")      # 估计处理组平均处理效应
st2.write("sources", "datasets", panel)
st2

StudyState(sources[1], design[2], variables[1], estimand[1]; 0 steps)

## Solving Donor Weights and Estimating ATT

`sv.tl.synthetic_control` first pivots the long panel into an outcome matrix of shape time × unit, then splits it at `treat_time=2015` into pre- and post-treatment periods. It then solves a constrained optimization (SLSQP) for a set of **non-negative weights that sum to one**, minimizing the mean squared error between the treated unit and its synthetic counterfactual **in the pre-treatment period**; this weighted combination of donors' trajectories is the counterfactual, and the average gap between treated and synthetic post-treatment is the ATT. Any unit that was **ever treated** is automatically excluded from the donor pool.

In [10]:
sv.tl.synthetic_control(st2, unit="firm_id", time="year",
                        treated_unit=0, treat_time=2015)  # 处理 firm_id=0, 从 2015 起

synth = st2.models["synth"]
pre_fit = st2.diagnostics["pre_fit"]
print(f"真实 att  = -0.8   (数据生成过程已知)")
print(f"估计 ATT  = {synth['att']:.4f}   (单个 treated 单位: firm_id=0)")
print(f"捐赠者数量 = {synth['n_donors']}   (已剔除曾被处理的单位)")
print(f"处理前拟合 = pre-RMSE {pre_fit['pre_rmse']:.4f}"
      f"  (前期 {pre_fit['n_pre']} 点, 后期 {pre_fit['n_post']} 点)")

真实 att  = -0.8   (数据生成过程已知)
估计 ATT  = -1.2829   (单个 treated 单位: firm_id=0)
捐赠者数量 = 20   (已剔除曾被处理的单位)
处理前拟合 = pre-RMSE 0.2350  (前期 5 点, 后期 3 点)


Non-negative weights summing to one mean the synthetic control is a **convex combination** of donors—"firm_id=0 ≈ a weighted average of these firms." Look at the largest donor contributors:

In [11]:
top_w = sorted(synth["weights"].items(), key=lambda kv: -kv[1])[:5]
pd.DataFrame(
    [{"donor_firm_id": int(d), "weight": round(w, 3)} for d, w in top_w]
)

,donor_firm_id,weight
0,39,0.453
1,31,0.267
2,20,0.140
3,36,0.128
4,26,0.012


The pre-treatment RMSE is only about 0.24, indicating the synthetic control closely tracked the true `firm_id=0` before 2015—the counterfactual stands on solid ground. But note: this **single-case** point estimate (ATT ≈ −1.28) is not precisely equal to the true value −0.8. The reason is straightforward: synthetic control addresses only **one** treated unit, has only 5 pre-treatment points, and donors themselves carry noise, so it lacks a conventional sampling distribution—single-case estimates are inherently **noisy**. Next we apply an honest check that recovery happens at the **aggregate** level.

## Single-Case Noise, Aggregate Recovery of True Value

Apply the same synthetic control workflow **to each** of the 20 treated units, collecting each unit's own ATT. Any individual unit will deviate from the true value, but if the data-generating process is honest, the **mean and median** of these 20 ATTs should return to −0.8.

In [12]:
treated_ids = sorted(panel.loc[panel["treat"] == 1, "firm_id"].unique())
per_unit_att = []
for u in treated_ids:
    s = sv.StudyState()
    s.write("design", "treatment", "treat"); s.write("design", "time", "year")
    s.write("variables", "outcome", "y"); s.write("estimand", "target", "ATT")
    s.write("sources", "datasets", panel)
    sv.tl.synthetic_control(s, unit="firm_id", time="year",
                            treated_unit=int(u), treat_time=2015)
    per_unit_att.append(s.models["synth"]["att"])

per_unit_att = np.array(per_unit_att, float)
print(f"真实 att           = -0.8")
print(f"单案例 (firm_id=0) = {synth['att']:.3f}    <- 有噪, 不精确")
print(f"处理组 ATT 均值     = {per_unit_att.mean():.3f}    <- 聚合复原真值")
print(f"处理组 ATT 中位数   = {np.median(per_unit_att):.3f}")
print(f"处理组 ATT 标准差   = {per_unit_att.std():.3f}   (n={per_unit_att.size} 个处理单位)")

真实 att           = -0.8
单案例 (firm_id=0) = -1.283    <- 有噪, 不精确
处理组 ATT 均值     = -0.837    <- 聚合复原真值
处理组 ATT 中位数   = -0.809
处理组 ATT 标准差   = 0.393   (n=20 个处理单位)


The single case `firm_id=0` gives ATT = −1.28, but the mean of 20 treated units is −0.84 and the median −0.81, both close to the true −0.8. This is honest behavior from synthetic control: it provides a credible counterfactual trajectory for an individual case, not a precise point estimate of the population effect; the true value emerges at the aggregate level.

## Synthetic Path Plot

Returning to the main case `firm_id=0`, `sv.pl.synth_path` reads the trajectories from `models.synth`, plotting the treated (solid line) and synthetic (dashed line) paths, with a vertical line at the treatment time. The two lines should nearly coincide pre-treatment (low pre-treatment RMSE), and their post-treatment divergence is the effect.

In [13]:
sv.pl.synth_path(st2, out="fig_synth.png")
print("已存:", st2.artifacts["figures"]["synth"]["path"])

已存: fig_synth.png


![Synthetic control path plot](fig_synth.png)

Before 2015, the treated and synthetic lines are nearly superimposed; afterwards, the treated systematically falls below the synthetic control, and this persistent downward gap is the estimated negative effect. The single-case gap size is noisy (approximately −1.3 here), but the direction is clear; the true value −0.8 is recovered by the aggregate estimate in the previous section.

---
## A Reproducible Evidence Chain

All the steps above are ordinary statistical estimation that any script can perform. What `socialverse` additionally does is record each step in an audit log within the research state: which function was used, what prerequisites it needs, what it outputs. Thus a completed analysis chain carries its own traceable **evidence spine**—in social science, "which step and which data does this conclusion come from" often matters as much as the conclusion itself. `st.summary()` prints out the slots and step counts for each chain.

In [14]:
print("=== 断点回归链 ===")
print(st.summary())
print("\n=== 合成控制链 ===")
print(st2.summary())

=== 断点回归链 ===
StudyState
  sources: datasets
  variables: outcome
  estimand: target
  models: rdd
  diagnostics: bandwidth
  artifacts: figures
  provenance: 2 step(s)

=== 合成控制链 ===
StudyState
  sources: datasets
  design: treatment, time
  variables: outcome
  estimand: target
  models: synth
  diagnostics: pre_fit
  artifacts: figures
  provenance: 2 step(s)


At finer granularity, you can unpack the RDD chain's provenance line by line, seeing what each step requires and produces. This contract is live: if a step's required inputs are missing, the call is blocked and errors before computation, rather than silently producing a wildly wrong result.

In [15]:
for rec in st.provenance:
    print(f"step {rec['step']}: {rec['function']}")
    print(f"    requires={rec['requires']}")
    print(f"    produces={rec['produces']}")

step 1: socialverse.tl._quasi.rdd
    requires={'sources': ['datasets'], 'variables': ['outcome'], 'estimand': ['target']}
    produces={'models': ['rdd'], 'diagnostics': ['bandwidth']}
step 2: socialverse.pl._figure2.rdd_plot
    requires={'models': ['rdd']}
    produces={'artifacts': ['figures']}


## Summary

We used two quasi-experimental tools to identify a causal effect each, and both recovered the known true values: regression discontinuity (matching `rdrobust`) estimated the discontinuity jump `τ̂ ≈ 2.99` using local linear fitting with triangular kernel, hitting the true value 3.0 precisely in a single pass; synthetic control (matching `Synth` / `gsynth` / `augsynth`) constructed counterfactuals using non-negative donor weights summing to one, with single-case noise but mean/median of 20 treated units ≈ −0.8, faithfully recovering the true value.

Compared to pure estimation tools, `socialverse` adds not faster algorithms but an analysis spine **with built-in evidence chains**: each method's prerequisites and outputs are machine-readable contracts that will actually stop you from proceeding, state and provenance run throughout, making conclusions traceable and reproducible. The next tutorial [12_psychometrics](12_psychometrics.ipynb) turns to measurement: how to estimate latent ability from a set of survey items using item response theory.